<a href="https://colab.research.google.com/github/Penugondanarasimha/AI_assisted_page_detector_py/blob/main/AI_assisted_page_detector_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Upload Excel only; any filename ending in .xlsx works.
from google.colab import files
uploaded = files.upload()
workbook = next((name for name in uploaded if name.lower().endswith('.xlsx')), None)
assert workbook, 'Please upload the original Volopay .xlsx file, not a ZIP.'

import pandas as pd
import numpy as np

# 1. Read the 6 Excel sheets (headers are in Excel row 4)
S = {s: pd.read_excel(workbook, sheet_name=s, header=3)
     for s in ['GSC','GA4','CRM','Content','Crawl','Backlinks']}

# 2. GSC: US non-brand clicks, impressions, ranking and CTR
q = S['GSC']
q = q[q.country.eq('United States') & ~q['query'].str.strip().str.lower().eq('volopay')].copy()
q['weighted_pos'] = q.impressions * q.avg_position
g = q.groupby('page').agg(impressions=('impressions','sum'), clicks=('clicks','sum'), weighted_pos=('weighted_pos','sum'))
g['position'] = g.weighted_pos / g.impressions
g['ctr'] = g.clicks / g.impressions
fit = q[(q.impressions>0)&(q.avg_position>0)].copy()
m, b = np.polyfit(np.log(fit.avg_position), np.log((fit.clicks/fit.impressions).clip(lower=.0001)), 1)
g['ctr_ratio'] = g.ctr / np.exp(b + m*np.log(g.position.clip(lower=1)))

# 3. GA4: organic conversion and broken form events
z = S['GA4']
a = z[z.default_channel.eq('Organic Search')].groupby('landing_page').agg(sessions=('sessions','sum'), submits=('form_submits','sum'))
a['submits_per_1k'] = 1000*a.submits/a.sessions.replace(0,np.nan)
z = z.assign(bad=z.form_submits.gt(z.form_starts), zero=z.form_submits.gt(0)&z.form_starts.eq(0))
a = a.join(z.groupby('landing_page').agg(bad_rows=('bad','sum'),zero_rows=('zero','sum')))

# 4. CRM: first-touch organic leads and product mismatch
c = S['CRM']; c = c[c.first_touch_source.eq('Organic Search')].copy()
c['misfit'] = c.disqual_reason.eq('Needs revolving credit line') | c.country.ne('United States')
r = c.dropna(subset=['first_landing_page']).groupby('first_landing_page').agg(
    leads=('lead_id','size'), misfit_share=('misfit','mean'),
    credit_dqs=('disqual_reason',lambda x:x.eq('Needs revolving credit line').sum()))

# 5. Combine data for each page; do not double-count Content and CRM pipeline
p = S['Content'].set_index('page').copy()
p['age'] = (pd.Timestamp('2026-06-30')-pd.to_datetime(p.last_updated)).dt.days
crawl = S['Crawl'].set_index('url')[['indexable','LCP_mobile_s']]
links = S['Backlinks'].query('brand=="Volopay"').set_index('page')[['page_ref_domains']].rename(columns={'page_ref_domains':'link_rds'})
p = p.join(g).join(a).join(r).join(crawl).join(links)
p['rds'] = p.link_rds.fillna(p.page_ref_domains)
money = p[p.page_type.eq('Money page')]
rd_base = money.page_ref_domains.median()
conv_base = a.reindex(money.index).submits_per_1k.median()
w = dict(V=.30,C=.10,K=.15,A=.20,T=.10,F=.15)
clip = lambda n: float(np.clip(n,0,1))

# 6. Six factors: visibility, CTR, conversion, authority, tracking, fit
# Missing factors are omitted rather than scored as perfect performance.
def score(x):
    V = clip((x.position-3)/15)*clip(np.log10(max(x.impressions,1))/4.5) if pd.notna(x.position) else np.nan
    C = clip(1-x.ctr_ratio) if pd.notna(x.ctr_ratio) else np.nan
    K = clip(1-x.submits_per_1k/conv_base) if x.page_type=='Money page' and pd.notna(x.submits_per_1k) else np.nan
    rd = clip(1-x.rds/rd_base) if x.page_type=='Money page' and pd.notna(x.rds) else np.nan
    age = clip((x.age-180)/540) if pd.notna(x.age) else np.nan
    speed = clip(x.LCP_mobile_s-2.5) if pd.notna(x.LCP_mobile_s) and x.LCP_mobile_s>0 else np.nan
    available = [i for i in [rd,age,speed] if pd.notna(i)]
    A = np.mean(available) if available else np.nan
    T = 1.0 if x.zero_rows>0 else .5 if x.bad_rows>=3 else 0.0
    F = clip(x.misfit_share) if pd.notna(x.leads) and x.leads>=5 else np.nan
    factors = dict(V=V,C=C,K=K,A=A,T=T,F=F)
    present = [k for k in w if pd.notna(factors[k])]
    return pd.Series({**factors,'score':round(100*sum(w[k]*factors[k] for k in present)/sum(w[k] for k in present),1)})

p = p.join(p.apply(score,axis=1))

# 7. Assign action, rank 8 pages, and export CSV
def action(x):
    if x['T']>=1: return 'FIX TRACKING FIRST'
    if pd.notna(x.F) and x.credit_dqs>=5 and x.F>=.5: return 'REPOSITION / CONSOLIDATE'
    if pd.notna(x.position) and x.position<=8.5: return 'DEFEND'
    if pd.notna(x.position) and x.position<=10 and x.C>.3: return 'IMPROVE CTR'
    if pd.notna(x.A) and x.A>=.5 and x.age>365: return 'REFRESH + BUILD AUTHORITY'
    if pd.notna(x.position) and x.position>10: return 'BUILD AUTHORITY / REPOSITION'
    return 'MONITOR'

p['action'] = p.apply(action,axis=1)
p['source_count'] = 1+p.indexable.notna().astype(int)+p.position.notna().astype(int)+p.sessions.notna().astype(int)+p.leads.notna().astype(int)+p.link_rds.notna().astype(int)
result = p[p.source_count>=3].sort_values(['score','impressions'],ascending=False)[['score','action','V','C','K','A','T','F','position','impressions','credit_dqs']]
print(result[['score','action']].head(8).to_string())
result.to_csv('Volopay_Part4_Scores.csv', index_label='page')


Saving Volopay_SEO_Candidate_Data_Pack (1) (1).xlsx to Volopay_SEO_Candidate_Data_Pack (1) (1) (3).xlsx
                                       score                        action
page                                                                      
/corporate-cards/                       59.3            FIX TRACKING FIRST
/business-credit-cards/                 45.2      REPOSITION / CONSOLIDATE
/spend-management/                      39.1  BUILD AUTHORITY / REPOSITION
/virtual-cards/                         30.4  BUILD AUTHORITY / REPOSITION
/accounts-payable-automation/           16.7                        DEFEND
/expense-management/                    13.2                        DEFEND
/blog/corporate-card-policy-template/   11.1                        DEFEND
/blog/what-is-a-corporate-card/          9.1                        DEFEND


In [ ]:
from google.colab import drive
drive.mount('/content/drive')